In [ ]:
import os
import openai
from openai import AzureOpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
endpoint = os.getenv("endpoint")
model_name = os.getenv("model_name")
deployment = os.getenv("deployement")
subscription_key = os.getenv("subscription_key")
api_version = os.getenv("api_version")
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)


In [ ]:
prompt_template = """
Calculate the total area of the manufacturing plant shown in this image.

STEPS TO FOLLOW:
1. Identify the scale reference in the image (scale bar, known object, or coordinate grid)
2. Convert pixel measurements to real-world units using the identified scale
3. Outline the complete boundary of the manufacturing plant. (look for darker outline)
4. Break down complex shapes into simpler geometric components if needed
5. Calculate the area of each component using appropriate formulas
6. Sum all component areas to determine the total plant area
7. Report the final area with appropriate units (square meters, square feet, etc.)

Please clearly indicate:
- The scale reference used and its real-world equivalent
- Any assumptions made about boundaries or measurements
- A breakdown of how different sections were measured briefly
- The final calculated area with proper units
"""
prompt_template_2 = """
You are analyzing a top-down (orthographic) image of a street scene to estimate the area of a building.

A yellow line is present in the image and labeled with a real-world distance (e.g., "20 meters"). 
This yellow line should be used to determine a pixel-to-meter conversion scale by comparing the real-world length with its pixel length in the image.

Your task is to:
1. Identify the yellow line and read the real-world distance labeled on it.
2. Measure the pixel length of this yellow line in the image.
3. Calculate the pixel-to-meter scale (i.e., how many meters per pixel).
4. Identify the main building in the image, and estimate its width and length in pixels.
5. Convert those dimensions into real-world meters using the calculated scale.
6. Compute the area in square meters.
7. Convert the area into square feet (1 m² = 10.7639 ft²).
8. Provide your final answer with all intermediate steps and assumptions clearly stated.

Do not guess or infer any information not visible in the image.
Avoid perspective-based assumptions — the view is orthographic.
Only use visible and measurable features in the image to estimate the dimensions.
"""


prompt_template_3 = """
You are looking at an image of a street scene from a top-down view. 
There are two yellow line in the image representoing two different road, and their real-world length are labeled directly on the image. 
Using those yellow line as a reference scale, estimate the area (in square feet) of the main building visible in the image. 
Clearly explain how you identified the scale, how you estimated the building's size using it, and provide the final area. 
Assume the view is orthographic with no perspective distortion.
Don't make random assumptions, make sure you are accurate in doing all the calculations.
"""

In [ ]:
import base64

# Load your image and encode it as base64
with open("images/cyber_park.png", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

response = client.chat.completions.create(
    model=deployment,  
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_template_2},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}"
                    }
                }
            ]
        }
    ],
    max_tokens=1000
)


******************
*******************

In [121]:
system_prompt = """
You are a precise image analysis assistant specializing in architectural measurements from aerial/satellite imagery. 
Your task is to calculate accurate building areas by using reference measurements in the image.
Always show your detailed calculation steps and use mathematical precision.
If you're uncertain about any measurements, acknowledge the uncertainty and provide a range rather than a single estimate.
"""

user_prompt = """
Please analyze this top-down (orthographic) image of a building and calculate its area precisely.

Important reference: A yellow line has been drawn on a road in the image, labeled as 16 meters wide.

Follow these exact steps:
1. Identify the yellow reference line showing 16 meters
2. Measure the pixel length of this yellow reference line
3. Calculate the precise pixel-to-meter conversion ratio
4. Identify the main building in the image
5. Measure the building's dimensions (width and length) in pixels
6. Convert these measurements to meters using your calculated ratio
7. Calculate the building's area in square meters
8. Convert to square feet (1 m² = 10.7639 ft²)

For each step, show your work with specific measurements and calculations.
If the building has an irregular shape, break it down into measurable rectangular sections.
If you notice any features that might affect the area calculation (like courtyards, extensions, etc.), account for those in your measurements.
"""

In [122]:
response = client.chat.completions.create(
    model=deployment,  
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}"
                    }
                }
            ]
        }
    ],
    max_tokens=1000
)

In [127]:
result_text = response.choices[0].message.content

# Save it to a text file
with open("cyber_park.txt", "w", encoding="utf-8") as file:
    file.write(result_text)

In [ ]:
# combined area of all towers. 
# actual floor area
64273 + 70130 + 69664 

204067

In [133]:
accuracy = ( 1- (227934 - 204067) / 204067) * 100
accuracy 

88.30433142056286

In [137]:
### Multiple Markings:

In [144]:
system_prompt = """
You are a highly specialized image analysis system for architectural measurements. Your primary function is to calculate building areas from aerial imagery with extreme precision.

Always prioritize reference measurements explicitly marked in the image. When analyzing aerial imagery, be aware that pixel density may be higher than you initially perceive - carefully count pixels at the highest possible resolution, and be especially mindful when measuring thin lines that represent known distances.

Approach all measurements methodically and consistently, showing your detailed work at each step.
"""

user_prompt = """
Calculate the precise area of the building in this top-down aerial image. One or more colored lines have been drawn on roads in the image with their real-world measurements labeled in meters.

CRITICAL MEASUREMENT INSTRUCTIONS:
1. First, locate all reference lines with labeled measurements in the image
2. For EACH reference line:
   a. Zoom in mentally to the maximum resolution
   b. Count pixels with extreme precision - reference lines are often thin but clearly visible
   c. Record the exact labeled measurement and corresponding pixel count
   d. Calculate meters-per-pixel ratio

3. USE ONLY THE MOST RELIABLE REFERENCE: If multiple references exist, compare their meters-per-pixel ratios. Choose the one with:
   - The clearest markings
   - The most precise measurement possibility
   - The least distortion from perspective
   - DO NOT average ratios unless they are within 1% of each other

4. For the building measurements:
   a. Identify the exact perimeter of the building
   b. Measure the full width and length in pixels, ensuring you capture the entire structure
   c. For irregular buildings, divide into regular sections and measure each separately
   d. Account for any courtyards or cutouts by subtracting their area

5. Convert your building measurements to real-world meters using ONLY your chosen reference ratio
6. Calculate the total area in square meters and square feet (1 m² = 10.7639 ft²)

IMPORTANT:
- You must show all numerical values for pixel counts, calculations, and conversions
- Aerial imagery requires extreme precision - challenge yourself to count pixels at the highest resolution
- Reference lines may be much shorter than building dimensions - this is normal and requires careful scaling
- Double-check all measurements before final calculation
"""

In [145]:
# Load your image and encode it as base64
with open("images/cyber_park2.png", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

response = client.chat.completions.create(
    model=deployment,  
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}"
                    }
                }
            ]
        }
    ],
    max_tokens=1000
)

In [146]:
result_text = response.choices[0].message.content

# Save it to a text file
with open("cyber_park2.txt", "w", encoding="utf-8") as file:
    file.write(result_text)